# Week 2 Example Solution — Notebook Workflow

This is a worked example showing how the notebook workflow should look when following the plan-first approach.

**Outer loop:** PLAN → EXECUTE → EVALUATE → DOCUMENT

**Inner loop:** ENGINEER → PLAN → GENERATE → VERIFY → REFINE

---

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

print("Libraries loaded successfully!")

In [ ]:
# === LOAD THE DATA ===
data = pd.read_csv("../data/lifestyle_mental_health.csv")

print(f"Dataset shape: {data.shape[0]} participants, {data.shape[1]} variables")
print(f"Columns: {list(data.columns)}")
print()
data.head(10)

## Step 1: Explore the Data

In [ ]:
print("Missing values per column:")
print(data.isnull().sum())
print()
data.describe().round(2)

### What do the numbers tell us?

The `describe()` output gives us the basics — means, standard deviations, ranges. But with 44 variables, it's hard to spot which ones are most related to depression just from summary statistics. We need to look at the *relationships between variables*.

Note: there are some missing values (~2-3%) in the lifestyle columns. That's normal in real data — people skip questions or sensors fail. We'll handle those when we get to plotting.

We asked the AI: *"Before I plan my visualisation, I want to understand which lifestyle variables are most related to DASS_Depression. Can you create a correlation matrix heatmap showing the Pearson correlations between the key lifestyle variables and DASS_Depression? Exclude the individual DASS items (DASS_1 through DASS_21) and other DASS subscales to avoid circular correlations. Use seaborn's heatmap with annotated values and a diverging colour map."*

In [ ]:
# === CORRELATION MATRIX HEATMAP ===
# This shows the Pearson correlation (r) between every pair of numeric variables.
# Values close to +1 or -1 indicate strong linear relationships.
# Values close to 0 indicate weak or no linear relationship.
#
# We exclude the individual DASS items (DASS_1 to DASS_21) and other DASS subscales
# because they're components of the depression score — correlating them would be circular.

key_vars = [
    "Sleep_hrs_night", "Exercise_hrs_week", "SocialMedia_hrs_week",
    "Screen_time_hrs_day", "Caffeine_cups_day", "Alcohol_drinks_week",
    "Study_hrs_week", "Social_Support_score", "Income_AUD",
    "Life_satisfaction", "DASS_Depression"
]
key_data = data[key_vars].dropna()
corr_matrix = key_data.corr().round(3)

fig_corr, ax_corr = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, square=True, linewidths=0.5,
    annot_kws={"size": 9}, ax=ax_corr
)
ax_corr.set_title("Correlation Matrix: Lifestyle Variables and Depression",
                   fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.show()

# Print the correlations with DASS_Depression, sorted by strength
print("\nCorrelations with DASS_Depression (sorted by absolute value):")
print("=" * 55)
dep_corr = corr_matrix["DASS_Depression"].drop("DASS_Depression")
dep_sorted = dep_corr.abs().sort_values(ascending=False)
for var in dep_sorted.index:
    r = dep_corr[var]
    strength = "strong" if abs(r) > 0.3 else "moderate" if abs(r) > 0.15 else "weak"
    print(f"  {var:.<35s} r = {r:+.3f}  ({strength})")

In [ ]:
# === PAIRPLOT: SCATTER GRID OF KEY VARIABLES ===
# A pairplot shows every combination of variables plotted against each other.
# This is a quick way to spot patterns, clusters, and outliers across the dataset.
# We'll focus on the variables most relevant to depression.

pairplot_vars = ["Sleep_hrs_night", "Exercise_hrs_week", "SocialMedia_hrs_week",
                 "Study_hrs_week", "DASS_Depression"]

pair_grid = sns.pairplot(
    data[pairplot_vars].dropna(), diag_kind="kde",
    plot_kws={"alpha": 0.15, "s": 8},
    height=2.2, aspect=1
)
pair_grid.figure.suptitle("Pairwise Relationships: Key Lifestyle Variables and Depression",
                          fontsize=14, fontweight="bold", y=1.02)
plt.show()

### What did exploration tell us?

The correlation heatmap and pairplot gave us a data-driven starting point for planning the visualisation:

- **Life satisfaction** has the strongest correlation with depression (r = −0.51) — but since it's also a mental health-adjacent measure, it's almost too obvious. More interesting to focus on the *lifestyle* variables.
- **Sleep** shows a strong negative association (r = −0.37) — more sleep, lower depression
- **Study hours** shows a strong positive association (r = +0.36) — probably a proxy for academic stress
- **Exercise** and **social support** are moderate protective factors (r ≈ −0.27 each)
- **Social media** and **screen time** show modest positive associations (r ≈ +0.14–0.16)
- **Height, weight, and siblings** show essentially zero correlation — these aren't worth including in our visualisation

There are also interesting inter-predictor correlations: social media correlates with screen time (naturally), and exercise correlates with sleep. This means some lifestyle factors cluster together.

This exploration step is important: instead of guessing which variables to plot, we used the data itself to guide our choices. The correlation table tells us *which* relationships are strongest, and the pairplot lets us see *what shape* those relationships have (linear? curved? clustered?).

## Step 2: PLAN

Now that we've explored the data, we have a much better sense of which variables to focus on. The correlation matrix showed that **sleep** (r = −0.37), **study hours** (r = +0.36), **exercise** (r = −0.27), and **social media** (r = +0.16) have the strongest lifestyle associations with depression.

We asked the AI: *"Based on our exploration of the dataset — where sleep (r = −0.37) and study hours (r = +0.36) showed the strongest lifestyle correlations with DASS_Depression, followed by exercise (r = −0.27) and social media (r = +0.16) — write an analysis plan for a 2×2 multi-panel scatter plot figure. The DASS_Depression score ranges from 0–42 (from the DASS-21 questionnaire). There are ~2-3% missing values in the lifestyle columns that need handling. Write the plan as a numbered list in markdown — what goes in each panel, what relationships we're testing, and what visual features to include. Don't write any code yet."*

Notice how the exploration findings made this prompt much more specific — we could tell the AI *which* variables and *why*.

Here's the plan we received and reviewed:

### AI-Generated Plan (reviewed and approved)

**Figure: 2×2 scatter plot grid exploring lifestyle factors and DASS depression scores**

1. **Panel 1 (top-left): Sleep vs Depression**
   - X-axis: `Sleep_hrs_night` (continuous, ~4–10)
   - Y-axis: `DASS_Depression` (integer, 0–42)
   - Expected relationship: Negative — more sleep associated with lower depression
   - Add: Linear trend line in red, Pearson r value

2. **Panel 2 (top-right): Exercise vs Depression**
   - X-axis: `Exercise_hrs_week` (continuous, 0–10)
   - Y-axis: `DASS_Depression`
   - Expected relationship: Negative — more exercise associated with lower depression
   - Add: Linear trend line in red, Pearson r value

3. **Panel 3 (bottom-left): Social Media vs Depression**
   - X-axis: `SocialMedia_hrs_week` (continuous, 0–50)
   - Y-axis: `DASS_Depression`
   - Expected relationship: Positive — more social media associated with higher depression
   - Add: Linear trend line in red, Pearson r value

4. **Panel 4 (bottom-right): Study Hours vs Depression**
   - X-axis: `Study_hrs_week` (continuous, 0–40)
   - Y-axis: `DASS_Depression`
   - Expected relationship: Positive — more study hours associated with higher depression (academic stress?)
   - Add: Linear trend line in red, Pearson r value

**Missing values:** Drop rows with NaN in the plotted variables before fitting trend lines.

**Styling:** Figure size 12×10, alpha=0.3 for scatter points, descriptive titles on each panel, overall figure title, `sns.despine()` for clean borders.

*We reviewed this plan and agreed with all four panel choices. Sleep and exercise are well-known protective factors. Social media is topical and interesting. Study hours is an unexpected finding — it might reflect academic pressure or time stress.*

## Step 3: ENGINEER and GENERATE

Based on our reviewed plan, we crafted a detailed prompt (ENGINEER) and asked the AI to generate the code (GENERATE):

> I have a pandas DataFrame called `data` with 3000 rows and 44 columns. It has columns including `Sleep_hrs_night` (continuous, 4–10), `Exercise_hrs_week` (continuous, 0–10), `SocialMedia_hrs_week` (continuous, 0–50), `Study_hrs_week` (continuous, 0–40), and `DASS_Depression` (integer, 0–42 — a depression score from the DASS-21 questionnaire). There are ~2-3% missing values in the lifestyle columns.
>
> Using matplotlib and seaborn, create a 2×2 figure (figsize 12×10) with:
> - Top-left: scatter plot of Sleep vs DASS_Depression with a linear trend line in red
> - Top-right: scatter plot of Exercise vs DASS_Depression with a linear trend line in red
> - Bottom-left: scatter plot of Social Media vs DASS_Depression with a linear trend line in red
> - Bottom-right: scatter plot of Study Hours vs DASS_Depression with a linear trend line in red
>
> Drop rows with missing values before plotting. Use alpha=0.3 for the scatter points, add descriptive titles to each subplot (fontsize=14), axis labels (fontsize=12), and use sns.despine() to clean up the borders. Add an overall figure title: "Lifestyle Factors and Depression: Exploring Key Relationships". Also add the Pearson correlation coefficient (r value) as text in the top-right corner of each panel.

Notice how specific this is — it mentions every column name, every panel, every styling detail, and how to handle missing values. This is what makes the AI produce good code on the first attempt.

In [ ]:
# === VISUALISATION CODE (from AI, first attempt) ===

# Define the panels: (column, title)
panels = [
    ("Sleep_hrs_night", "Sleep (hrs/night) vs Depression"),
    ("Exercise_hrs_week", "Exercise (hrs/week) vs Depression"),
    ("SocialMedia_hrs_week", "Social Media (hrs/week) vs Depression"),
    ("Study_hrs_week", "Study Hours (hrs/week) vs Depression"),
]

# Drop rows with missing values in our variables of interest
plot_cols = [col for col, _ in panels] + ["DASS_Depression"]
data_clean = data.dropna(subset=plot_cols)
print(f"Plotting with {len(data_clean)} participants (after dropping missing values)")

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (col, title) in enumerate(panels):
    row, col_idx = divmod(idx, 2)
    ax = axes[row, col_idx]

    # Scatter plot
    sns.scatterplot(data=data_clean, x=col, y="DASS_Depression", alpha=0.3, s=20,
                    color="#4C72B0", ax=ax)

    # Trend line
    x_vals = data_clean[col].values
    y_vals = data_clean["DASS_Depression"].values
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax.plot(x_line, p(x_line), color="red", linewidth=2.5, alpha=0.8)

    # Correlation coefficient
    r = np.corrcoef(x_vals, y_vals)[0, 1]
    ax.text(0.95, 0.95, f"r = {r:.3f}",
            transform=ax.transAxes, fontsize=12, fontweight="bold",
            ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="gray", alpha=0.8))

    # Labels and title
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel(col.replace("_", " "), fontsize=12)
    ax.set_ylabel("DASS Depression Score", fontsize=12)

# Overall title and cleanup
fig.suptitle("Lifestyle Factors and Depression: Exploring Key Relationships",
             fontsize=18, fontweight="bold", y=1.02)
sns.despine()
plt.tight_layout()
plt.show()

## Step 4: VERIFY and REFINE

The code ran on the first attempt — the strong prompt paid off. Let's check:

- [x] **Runs without errors** — yes
- [x] **Axis labels correct** — yes, all four panels show the right variables
- [x] **Patterns make sense** — Sleep and exercise show negative associations (as expected), social media and study hours show positive associations
- [x] **Figure is readable** — fonts are large enough, panels are well-spaced
- [x] **Looks professional** — clean, consistent styling across all panels

The r values confirm what we saw in the exploration:
- Sleep has a strong negative relationship with depression (r = −0.37) — the strongest lifestyle predictor
- Study hours shows a comparable positive association (r = +0.36) — probably reflecting academic stress
- Exercise shows a moderate protective effect (r = −0.27)
- Social media shows a weaker but still visible positive association (r = +0.16)

These correlations are realistic for psychological data — in the real world, most lifestyle-mental health correlations fall in the r = 0.15–0.40 range. The scatter plots show plenty of individual variation around the trend lines, which is also realistic.

No refinement needed — the plan-first approach meant our prompt was precise enough to get a good result immediately. And because we explored the data first, we weren't surprised by any of these patterns.

In [ ]:
# === SAVE THE FIGURE ===
import os
os.makedirs("images", exist_ok=True)

fig.savefig("images/our_visualisation.png", dpi=300, bbox_inches="tight")
print("Figure saved as 'images/our_visualisation.png'")

## Step 5: DOCUMENT and Reflect

**What relationships did we explore, and what did we find?**

We explored four lifestyle factors and their associations with DASS depression scores. Sleep had the strongest negative association (r = −0.37) — more sleep was associated with lower depression. Study hours showed a comparable positive association (r = +0.36) — possibly reflecting academic stress. Exercise was also protective (r = −0.27), and social media use showed a modest positive association (r = +0.16).

**What was the most interesting pattern in the data?**

The study hours finding was unexpected. We might assume studying is a neutral or positive activity, but the data suggests that people who study more also report higher depression scores. This could reflect academic pressure, or it could be that students (who study more) are a group with higher depression rates in general. The dataset includes an Occupation variable — it would be worth checking whether this relationship holds within occupation groups.

**What did we learn about prompting an LLM?**

- **Exploring the data first was the biggest win** — it gave us specific r values and variable names to put in our prompts
- Mentioning the DASS-21 score range (0–42) and the missing values helped the AI produce appropriate code
- Describing each panel individually (instead of saying "make a figure") meant the AI knew exactly what to build
- We only needed one prompt iteration — zero refinements needed

**Did having a written plan before coding help? How?**

Absolutely. The plan forced us to decide what goes in each panel before writing any code. But even more useful was the **exploration step before the plan** — without the correlation matrix, we might have included variables like caffeine or screen time that turned out to have weaker associations. The exploration → plan → code pipeline meant every step was informed by the previous one.

**What would we do differently next time?**

We'd explore the categorical variables too — does depression differ by Occupation or Relationship_status? The dataset has these columns but we focused only on continuous variables. We'd also compare DASS_Depression with DASS_Anxiety and DASS_Stress to see whether the same lifestyle factors predict all three, or whether they have different profiles.

In [ ]:
# === SAVE ALL FIGURES ===
os.makedirs("images", exist_ok=True)

fig_corr.savefig("images/correlation_heatmap.png", dpi=300, bbox_inches="tight")
pair_grid.savefig("images/pairplot.png", dpi=300, bbox_inches="tight")
print("Saved 'images/correlation_heatmap.png' and 'images/pairplot.png'")

---

## Bonus Challenges

Finished the main task early? Here are some extensions we tried. Each one builds on what we've already done and practises a different visualisation skill.

### Bonus 1: Group Comparison — Depression by Occupation

The correlation analysis focused on continuous variables. But the dataset also has categorical variables like Occupation. We asked the AI: *"Create a figure with violin plots comparing DASS_Depression scores across Occupation groups. Order the groups by their median depression score. Use a colour palette that's easy to distinguish."*

In [ ]:
# === BONUS 1: DEPRESSION BY OCCUPATION ===

# Order occupation groups by median depression score
occ_order = (data.groupby("Occupation")["DASS_Depression"]
             .median().sort_values().index.tolist())

fig_bonus, ax_bonus = plt.subplots(figsize=(12, 6))

sns.violinplot(
    data=data, x="Occupation", y="DASS_Depression",
    order=occ_order, palette="Set2",
    inner="quart", ax=ax_bonus
)

# Add individual points for context
sns.stripplot(
    data=data, x="Occupation", y="DASS_Depression",
    order=occ_order, color="black", alpha=0.05, s=2,
    jitter=True, ax=ax_bonus
)

ax_bonus.set_xlabel("", fontsize=12)
ax_bonus.set_ylabel("DASS Depression Score", fontsize=12)
ax_bonus.set_title("Depression Scores by Occupation Group",
                   fontsize=16, fontweight="bold")

# Add sample sizes below each group
for i, occ in enumerate(occ_order):
    n = len(data[data["Occupation"] == occ])
    median = data[data["Occupation"] == occ]["DASS_Depression"].median()
    ax_bonus.text(i, -3, f"n={n}", ha="center", fontsize=9, color="grey")

sns.despine()
plt.tight_layout()
plt.show()

fig_bonus.savefig("images/bonus_group_comparison.png", dpi=300, bbox_inches="tight")
print("Saved 'images/bonus_group_comparison.png'")

### Bonus 2: Annotated Figure — Highlighting Key Findings

We asked the AI: *"Take our 2×2 scatter figure and add text annotations pointing out the key findings — the strongest lifestyle predictor (sleep), the unexpected study hours finding, and the social media association. Use arrows or text boxes to draw attention to the most important patterns."*

In [ ]:
# === BONUS 2: ANNOTATED SCATTER PLOT FIGURE ===
# Same 2x2 figure as before, but with annotations highlighting key findings

fig_ann, axes_ann = plt.subplots(2, 2, figsize=(12, 10))

panels = [
    ("Sleep_hrs_night", "Sleep (hrs/night) vs Depression"),
    ("Exercise_hrs_week", "Exercise (hrs/week) vs Depression"),
    ("SocialMedia_hrs_week", "Social Media (hrs/week) vs Depression"),
    ("Study_hrs_week", "Study Hours (hrs/week) vs Depression"),
]

for idx, (col, title) in enumerate(panels):
    row, col_idx = divmod(idx, 2)
    ax = axes_ann[row, col_idx]
    sns.scatterplot(data=data_clean, x=col, y="DASS_Depression", alpha=0.3, s=20,
                    color="#4C72B0", ax=ax)
    x_vals = data_clean[col].values
    y_vals = data_clean["DASS_Depression"].values
    z = np.polyfit(x_vals, y_vals, 1)
    p = np.poly1d(z)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 200)
    ax.plot(x_line, p(x_line), color="red", linewidth=2.5, alpha=0.8)
    r = np.corrcoef(x_vals, y_vals)[0, 1]
    ax.text(0.95, 0.95, f"r = {r:.3f}", transform=ax.transAxes, fontsize=12,
            fontweight="bold", ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor="gray", alpha=0.8))
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel(col.replace("_", " "), fontsize=12)
    ax.set_ylabel("DASS Depression Score", fontsize=12)

# Annotation: Sleep panel — strongest lifestyle predictor
axes_ann[0, 0].annotate(
    "Strongest lifestyle\npredictor (r = \u22120.37)",
    xy=(9, 5), xytext=(7, 35),
    fontsize=10, fontweight="bold", color="#C0392B",
    arrowprops=dict(arrowstyle="->", color="#C0392B", lw=2),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#FADBD8", edgecolor="#C0392B")
)

# Annotation: Social media panel — modest risk factor
axes_ann[1, 0].annotate(
    "More social media \u2192\nslightly higher depression",
    xy=(40, 25), xytext=(10, 38),
    fontsize=10, fontweight="bold", color="#8E44AD",
    arrowprops=dict(arrowstyle="->", color="#8E44AD", lw=2),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#F5EEF8", edgecolor="#8E44AD")
)

# Annotation: Study hours panel — unexpected finding
axes_ann[1, 1].annotate(
    "Unexpected: more study\nhours \u2192 higher depression",
    xy=(30, 30), xytext=(5, 38),
    fontsize=10, fontweight="bold", color="#2E86C1",
    arrowprops=dict(arrowstyle="->", color="#2E86C1", lw=2),
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#D6EAF8", edgecolor="#2E86C1")
)

fig_ann.suptitle("Lifestyle Factors and Depression \u2014 Annotated Key Findings",
                 fontsize=18, fontweight="bold", y=1.02)
sns.despine()
plt.tight_layout()
plt.show()

fig_ann.savefig("images/bonus_annotated.png", dpi=300, bbox_inches="tight")
print("Saved 'images/bonus_annotated.png'")